# Sistema de Gestão de Confeitaria (Banco de Dados PostgreSQL)
**Integrantes:** Pedro de Lima Talasca e Weslei  
**Objetivo:** Implementação de conexão com PostgreSQL via `psycopg2`, modelagem orientada a objetos, operações CRUD completas para todas as entidades (`cliente`, `cliente_telefone`, `tipo_doce`, `doce`, `pedido`, `item_pedido` e `solicitacao`), tratamento de exceções (como violação de restrições `UNIQUE`) e menu interativo.

## 1. Instalação e Conexão com o Banco de Dados
Nesta seção instalamos a biblioteca `psycopg2` e estabelecemos a conexão com o banco `confeitaria` no PostgreSQL.

In [84]:
import psycopg2 as pg

conexao = pg.connect(
    dbname="confeitaria",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)
cursor = conexao.cursor()
print("Conectado ao banco de dados com sucesso!")

Conectado ao banco de dados com sucesso!


## 2. Definição das Classes de Modelo
Classes Python com tipagem explícita (`type hints`) representando cada tabela do banco de dados. Os campos `cep` e `numero` (telefone) foram definidos como `str` (`VARCHAR` no banco) para evitar perdas de formatação ou estouro de inteiro.

In [85]:
class Cliente:
    def __init__(self, nome: str, logradouro: str, numero: int, bairro: str, cidade: str, cep: str, id_cliente: int | None = None):
        self.id_cliente = id_cliente
        self.nome = nome
        self.logradouro = logradouro
        self.numero = numero
        self.bairro = bairro
        self.cidade = cidade
        self.cep = cep

    def __str__(self) -> str:
        return f"Cliente(id={self.id_cliente}, nome='{self.nome}', endereco='{self.logradouro}, {self.numero} - {self.bairro}, {self.cidade} ({self.cep})')"

    def __repr__(self) -> str:
        return self.__str__()


class ClienteTelefone:
    def __init__(self, id_cliente: int, numero: str, id_telefone: int | None = None):
        self.id_telefone = id_telefone
        self.id_cliente = id_cliente
        self.numero = numero

    def __str__(self) -> str:
        return f"ClienteTelefone(id={self.id_telefone}, id_cliente={self.id_cliente}, numero='{self.numero}')"

    def __repr__(self) -> str:
        return self.__str__()


class TipoDoce:
    def __init__(self, sabor: str, tipo: str, id_tipo: int | None = None):
        self.id_tipo = id_tipo
        self.sabor = sabor
        self.tipo = tipo

    def __str__(self) -> str:
        return f"TipoDoce(id={self.id_tipo}, tipo='{self.tipo}', sabor='{self.sabor}')"

    def __repr__(self) -> str:
        return self.__str__()


class Doce:
    def __init__(self, id_tipo: int, preco: float, id_doce: int | None = None):
        self.id_doce = id_doce
        self.id_tipo = id_tipo
        self.preco = preco

    def __str__(self) -> str:
        return f"Doce(id={self.id_doce}, id_tipo={self.id_tipo}, preco=R${self.preco:.2f})"

    def __repr__(self) -> str:
        return self.__str__()


class Pedido:
    def __init__(self, data_feita: str, data_prevista: str, data_entregue: str | None = None, id_pedido: int | None = None):
        self.id_pedido = id_pedido
        self.data_feita = data_feita
        self.data_prevista = data_prevista
        self.data_entregue = data_entregue

    def __str__(self) -> str:
        return f"Pedido(id={self.id_pedido}, feita='{self.data_feita}', prevista='{self.data_prevista}', entregue='{self.data_entregue}')"

    def __repr__(self) -> str:
        return self.__str__()


class ItemPedido:
    def __init__(self, id_pedido: int, id_doce: int, id_cliente: int, total_pago: float, troco: float):
        self.id_pedido = id_pedido
        self.id_doce = id_doce
        self.id_cliente = id_cliente
        self.total_pago = total_pago
        self.troco = troco

    def __str__(self) -> str:
        return f"ItemPedido(pedido={self.id_pedido}, doce={self.id_doce}, cliente={self.id_cliente}, pago=R${self.total_pago:.2f}, troco=R${self.troco:.2f})"

    def __repr__(self) -> str:
        return self.__str__()


class Solicitacao:
    def __init__(self, id_cliente: int, id_pedido: int):
        self.id_cliente = id_cliente
        self.id_pedido = id_pedido

    def __str__(self) -> str:
        return f"Solicitacao(id_cliente={self.id_cliente}, id_pedido={self.id_pedido})"

    def __repr__(self) -> str:
        return self.__str__()

## 3. Funções CRUD por Tabela
Abaixo estão implementadas as quatro operações básicas (**Criar**, **Ler**, **Atualizar** e **Deletar**) para todas as 7 tabelas do banco, utilizando `RETURNING` nos comandos `INSERT` com chaves `SERIAL` e `conexao.rollback()` devidamente recuado dentro dos blocos `except`.

### 3.1 CRUD — Tabela `cliente`

In [91]:
def criar_cliente(nome: str, logradouro: str, numero: int, bairro: str, cidade: str, cep: str) -> Cliente | None:
    try:
        cursor.execute(
            "INSERT INTO cliente (nome, logradouro, numero, bairro, cidade, cep) "
            "VALUES (%s, %s, %s, %s, %s, %s) RETURNING id_cliente;",
            (nome, logradouro, numero, bairro, cidade, cep)
        )
        id_cliente = cursor.fetchone()[0]
        conexao.commit()
        return Cliente(nome, logradouro, numero, bairro, cidade, cep, id_cliente=id_cliente)
    except pg.DatabaseError as e:
        print(f"Erro ao criar cliente: {e}")
        conexao.rollback()
        return None

def ler_clientes() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM cliente ORDER BY id_cliente;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler clientes: {e}")
        conexao.rollback()
        return []

def atualizar_cliente(id_cliente: int, novo_logradouro: str, novo_numero: int, novo_bairro: str, nova_cidade: str, novo_cep: str) -> bool:
    try:
        cursor.execute(
            "UPDATE cliente SET logradouro = %s, numero = %s, bairro = %s, cidade = %s, cep = %s WHERE id_cliente = %s;",
            (novo_logradouro, novo_numero, novo_bairro, nova_cidade, novo_cep, id_cliente)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar cliente: {e}")
        conexao.rollback()
        return False

def deletar_cliente(id_cliente: int) -> bool:
    try:
        cursor.execute("DELETE FROM cliente WHERE id_cliente = %s;", (id_cliente,))
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar cliente: {e}")
        conexao.rollback()
        return False

### 3.2 CRUD — Tabela `cliente_telefone`

In [92]:
def criar_telefone(id_cliente: int, numero: str) -> ClienteTelefone | None:
    try:
        cursor.execute(
            "INSERT INTO cliente_telefone (id_cliente, numero) VALUES (%s, %s) RETURNING id_telefone;",
            (id_cliente, numero)
        )
        id_telefone = cursor.fetchone()[0]
        conexao.commit()
        return ClienteTelefone(id_cliente, numero, id_telefone=id_telefone)
    except pg.DatabaseError as e:
        print(f"Erro ao criar telefone: {e}")
        conexao.rollback()
        return None

def ler_telefones() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM cliente_telefone ORDER BY id_telefone;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler telefones: {e}")
        conexao.rollback()
        return []

def atualizar_telefone(id_telefone: int, novo_numero: str) -> bool:
    try:
        cursor.execute(
            "UPDATE cliente_telefone SET numero = %s WHERE id_telefone = %s;",
            (novo_numero, id_telefone)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar telefone: {e}")
        conexao.rollback()
        return False

def deletar_telefone(id_telefone: int) -> bool:
    try:
        cursor.execute("DELETE FROM cliente_telefone WHERE id_telefone = %s;", (id_telefone,))
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar telefone: {e}")
        conexao.rollback()
        return False

### 3.3 CRUD — Tabela `tipo_doce`

In [93]:
def criar_tipo_doce(sabor: str, tipo: str) -> TipoDoce | None:
    try:
        cursor.execute(
            "INSERT INTO tipo_doce (sabor, tipo) VALUES (%s, %s) RETURNING id_tipo;",
            (sabor, tipo)
        )
        id_tipo = cursor.fetchone()[0]
        conexao.commit()
        return TipoDoce(sabor, tipo, id_tipo=id_tipo)
    except pg.DatabaseError as e:
        print(f"Erro ao criar tipo de doce: {e}")
        conexao.rollback()
        return None

def ler_tipos_doce() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM tipo_doce ORDER BY id_tipo;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler tipos de doce: {e}")
        conexao.rollback()
        return []

def atualizar_tipo_doce(id_tipo: int, novo_sabor: str, novo_tipo: str) -> bool:
    try:
        cursor.execute(
            "UPDATE tipo_doce SET sabor = %s, tipo = %s WHERE id_tipo = %s;",
            (novo_sabor, novo_tipo, id_tipo)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar tipo de doce: {e}")
        conexao.rollback()
        return False

def deletar_tipo_doce(id_tipo: int) -> bool:
    try:
        cursor.execute("DELETE FROM tipo_doce WHERE id_tipo = %s;", (id_tipo,))
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar tipo de doce: {e}")
        conexao.rollback()
        return False

### 3.4 CRUD — Tabela `doce`

In [94]:
def criar_doce(id_tipo: int, preco: float) -> Doce | None:
    try:
        cursor.execute(
            "INSERT INTO doce (id_tipo, preco) VALUES (%s, %s) RETURNING id_doce;",
            (id_tipo, preco)
        )
        id_doce = cursor.fetchone()[0]
        conexao.commit()
        return Doce(id_tipo, preco, id_doce=id_doce)
    except pg.DatabaseError as e:
        print(f"Erro ao criar doce: {e}")
        conexao.rollback()
        return None

def ler_doces() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM doce ORDER BY id_doce;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler doces: {e}")
        conexao.rollback()
        return []

def atualizar_doce(id_doce: int, novo_preco: float) -> bool:
    try:
        cursor.execute(
            "UPDATE doce SET preco = %s WHERE id_doce = %s;",
            (novo_preco, id_doce)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar doce: {e}")
        conexao.rollback()
        return False

def deletar_doce(id_doce: int) -> bool:
    try:
        cursor.execute("DELETE FROM doce WHERE id_doce = %s;", (id_doce,))
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar doce: {e}")
        conexao.rollback()
        return False

### 3.5 CRUD — Tabela `pedido`

In [95]:
def criar_pedido(data_feita: str, data_prevista: str, data_entregue: str | None = None) -> Pedido | None:
    try:
        cursor.execute(
            "INSERT INTO pedido (data_feita, data_prevista, data_entregue) VALUES (%s, %s, %s) RETURNING id_pedido;",
            (data_feita, data_prevista, data_entregue)
        )
        id_pedido = cursor.fetchone()[0]
        conexao.commit()
        return Pedido(data_feita, data_prevista, data_entregue, id_pedido=id_pedido)
    except pg.DatabaseError as e:
        print(f"Erro ao criar pedido: {e}")
        conexao.rollback()
        return None

def ler_pedidos() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM pedido ORDER BY id_pedido;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler pedidos: {e}")
        conexao.rollback()
        return []

def atualizar_pedido(id_pedido: int, nova_data_entregue: str) -> bool:
    try:
        cursor.execute(
            "UPDATE pedido SET data_entregue = %s WHERE id_pedido = %s;",
            (nova_data_entregue, id_pedido)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar pedido: {e}")
        conexao.rollback()
        return False

def deletar_pedido(id_pedido: int) -> bool:
    try:
        cursor.execute("DELETE FROM pedido WHERE id_pedido = %s;", (id_pedido,))
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar pedido: {e}")
        conexao.rollback()
        return False

### 3.6 CRUD — Tabela `item_pedido`

In [96]:
def criar_item_pedido(id_pedido: int, id_doce: int, id_cliente: int, total_pago: float, troco: float) -> ItemPedido | None:
    try:
        cursor.execute(
            "INSERT INTO item_pedido (id_pedido, id_doce, id_cliente, total_pago, troco) "
            "VALUES (%s, %s, %s, %s, %s) RETURNING id_pedido, id_doce, id_cliente;",
            (id_pedido, id_doce, id_cliente, total_pago, troco)
        )
        resultado = cursor.fetchone()
        conexao.commit()
        return ItemPedido(resultado[0], resultado[1], resultado[2], total_pago, troco)
    except pg.DatabaseError as e:
        print(f"Erro ao criar item do pedido: {e}")
        conexao.rollback()
        return None

def ler_itens_pedido() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM item_pedido ORDER BY id_pedido, id_doce;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler itens do pedido: {e}")
        conexao.rollback()
        return []

def atualizar_item_pedido(id_pedido: int, id_doce: int, id_cliente: int, novo_total_pago: float, novo_troco: float) -> bool:
    try:
        cursor.execute(
            "UPDATE item_pedido SET total_pago = %s, troco = %s "
            "WHERE id_pedido = %s AND id_doce = %s AND id_cliente = %s;",
            (novo_total_pago, novo_troco, id_pedido, id_doce, id_cliente)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar item do pedido: {e}")
        conexao.rollback()
        return False

def deletar_item_pedido(id_pedido: int, id_doce: int, id_cliente: int) -> bool:
    try:
        cursor.execute(
            "DELETE FROM item_pedido WHERE id_pedido = %s AND id_doce = %s AND id_cliente = %s;",
            (id_pedido, id_doce, id_cliente)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar item do pedido: {e}")
        conexao.rollback()
        return False

### 3.7 CRUD — Tabela `solicitacao`

In [97]:
def criar_solicitacao(id_cliente: int, id_pedido: int) -> Solicitacao | None:
    try:
        cursor.execute(
            "INSERT INTO solicitacao (id_cliente, id_pedido) VALUES (%s, %s) RETURNING id_cliente, id_pedido;",
            (id_cliente, id_pedido)
        )
        resultado = cursor.fetchone()
        conexao.commit()
        return Solicitacao(resultado[0], resultado[1])
    except pg.DatabaseError as e:
        print(f"Erro ao criar solicitação: {e}")
        conexao.rollback()
        return None

def ler_solicitacoes() -> list[tuple]:
    try:
        cursor.execute("SELECT * FROM solicitacao ORDER BY id_cliente, id_pedido;")
        return cursor.fetchall()
    except pg.DatabaseError as e:
        print(f"Erro ao ler solicitações: {e}")
        conexao.rollback()
        return []

def atualizar_solicitacao(id_cliente_atual: int, id_pedido_atual: int, novo_id_cliente: int, novo_id_pedido: int) -> bool:
    try:
        cursor.execute(
            "UPDATE solicitacao SET id_cliente = %s, id_pedido = %s WHERE id_cliente = %s AND id_pedido = %s;",
            (novo_id_cliente, novo_id_pedido, id_cliente_atual, id_pedido_atual)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao atualizar solicitação: {e}")
        conexao.rollback()
        return False

def deletar_solicitacao(id_cliente: int, id_pedido: int) -> bool:
    try:
        cursor.execute(
            "DELETE FROM solicitacao WHERE id_cliente = %s AND id_pedido = %s;",
            (id_cliente, id_pedido)
        )
        conexao.commit()
        return cursor.rowcount > 0
    except pg.DatabaseError as e:
        print(f"Erro ao deletar solicitação: {e}")
        conexao.rollback()
        return False

## 4. Como Usar — Exemplos Práticos por Tabela
As células abaixo demonstram o ciclo completo (Criar, Ler, Atualizar e Deletar) em cada tabela do sistema.

### 4.1 Como usar: `cliente` e `cliente_telefone`

In [98]:
# 1. Criar clientes (com CEP formatado corretamente como string)
c1 = criar_cliente("Paulo", "Avenida Borges de Medeiros", 123, "Centro", "Porto Alegre", "90020-025")
c2 = criar_cliente("Maria", "Rua Paulista", 456, "Bela Vista", "São Paulo", "01310-100")
print("Criados:", c1, "|", c2)

# 2. Ler clientes
print("Lista de clientes:", ler_clientes())

# 3. Atualizar cliente
if c1:
    atualizar_cliente(c1.id_cliente, "Rua da Praia", 999, "Centro Histórico", "Porto Alegre", "90010-150")
    print("Após atualização:", ler_clientes())

# 4. Criar, ler, atualizar e deletar telefone (com DDD como VARCHAR)
if c1:
    tel = criar_telefone(c1.id_cliente, "51999998888")
    print("Telefone criado:", tel)
    print("Telefones no banco:", ler_telefones())
    if tel:
        atualizar_telefone(tel.id_telefone, "51988887777")
        print("Telefone atualizado:", ler_telefones())
        deletar_telefone(tel.id_telefone)
        print("Após deletar telefone:", ler_telefones())

Criados: Cliente(id=10, nome='Paulo', endereco='Avenida Borges de Medeiros, 123 - Centro, Porto Alegre (90020-025)') | Cliente(id=11, nome='Maria', endereco='Rua Paulista, 456 - Bela Vista, São Paulo (01310-100)')
Lista de clientes: [(1, 'Paulo', 'Avenida', 123, 'Rosas Marias', 'POA', '11667'), (2, 'Maria', 'Avenida', 456, 'Flores Azuis', 'SP', '11889'), (3, 'João', 'Avenida', 789, 'Maças Vermelhas', 'RJ', '4246'), (10, 'Paulo', 'Avenida Borges de Medeiros', 123, 'Centro', 'Porto Alegre', '90020-025'), (11, 'Maria', 'Rua Paulista', 456, 'Bela Vista', 'São Paulo', '01310-100')]
Após atualização: [(1, 'Paulo', 'Avenida', 123, 'Rosas Marias', 'POA', '11667'), (2, 'Maria', 'Avenida', 456, 'Flores Azuis', 'SP', '11889'), (3, 'João', 'Avenida', 789, 'Maças Vermelhas', 'RJ', '4246'), (10, 'Paulo', 'Rua da Praia', 999, 'Centro Histórico', 'Porto Alegre', '90010-150'), (11, 'Maria', 'Rua Paulista', 456, 'Bela Vista', 'São Paulo', '01310-100')]
Erro ao criar telefone: ERRO:  o valor "51999998888

### 4.2 Como usar: `tipo_doce` e `doce` (incluindo teste de restrição `UNIQUE`)

In [99]:
# 1. Criar tipos de doce
td1 = criar_tipo_doce("Limão", "Torta")
td2 = criar_tipo_doce("Chocolate", "Bolo")
print("Tipos criados:", td1, "|", td2)

# 2. Teste de erro: tentando inserir par (sabor, tipo) duplicado (viola restrição UNIQUE)
print("\n--- Testando inserção inválida (UNIQUE duplicado) ---")
duplicado = criar_tipo_doce("Limão", "Torta")
print("Retorno esperado (None):", duplicado)

# 3. Criar doce vinculado ao id_tipo
if td1:
    doce1 = criar_doce(td1.id_tipo, 45.00)
    print("\nDoce criado:", doce1)
    print("Doces no banco:", ler_doces())
    if doce1:
        atualizar_doce(doce1.id_doce, 49.90)
        print("Doce após reajuste de preço:", ler_doces())

Tipos criados: TipoDoce(id=10, tipo='Torta', sabor='Limão') | TipoDoce(id=11, tipo='Bolo', sabor='Chocolate')

--- Testando inserção inválida (UNIQUE duplicado) ---
Retorno esperado (None): TipoDoce(id=12, tipo='Torta', sabor='Limão')

Doce criado: Doce(id=10, id_tipo=10, preco=R$45.00)
Doces no banco: [(7, Decimal('10.00'), 1), (8, Decimal('5.00'), 2), (9, Decimal('15.00'), 3), (10, Decimal('45.00'), 10)]
Doce após reajuste de preço: [(7, Decimal('10.00'), 1), (8, Decimal('5.00'), 2), (9, Decimal('15.00'), 3), (10, Decimal('49.90'), 10)]


### 4.3 Como usar: `pedido`, `item_pedido` e `solicitacao`

In [100]:
# 1. Criar pedido
ped1 = criar_pedido("2026-11-17", "2026-11-20", None)
print("Pedido criado:", ped1)

# 2. Vincular solicitação e item do pedido
if c1 and ped1 and doce1:
    sol1 = criar_solicitacao(c1.id_cliente, ped1.id_pedido)
    print("Solicitação criada:", sol1)
    print("Solicitações no banco:", ler_solicitacoes())

    item1 = criar_item_pedido(ped1.id_pedido, doce1.id_doce, c1.id_cliente, 50.00, 0.10)
    print("Item de pedido criado:", item1)
    print("Itens no banco:", ler_itens_pedido())

    # Atualizar pagamento do item e data de entrega do pedido
    atualizar_item_pedido(ped1.id_pedido, doce1.id_doce, c1.id_cliente, 60.00, 10.10)
    atualizar_pedido(ped1.id_pedido, "2026-11-20")
    print("Pedido entregue:", ler_pedidos())
    print("Item atualizado:", ler_itens_pedido())

Pedido criado: Pedido(id=10, feita='2026-11-17', prevista='2026-11-20', entregue='None')
Solicitação criada: Solicitacao(id_cliente=10, id_pedido=10)
Solicitações no banco: [(10, 10)]
Item de pedido criado: ItemPedido(pedido=10, doce=10, cliente=10, pago=R$50.00, troco=R$0.10)
Itens no banco: [(Decimal('50.00'), Decimal('0.10'), 10, 10, 10)]
Pedido entregue: [(1, datetime.date(2026, 6, 12), datetime.date(2026, 6, 20), datetime.date(2026, 6, 21)), (2, datetime.date(2026, 7, 13), datetime.date(2026, 7, 21), datetime.date(2026, 7, 22)), (3, datetime.date(2026, 8, 14), datetime.date(2026, 8, 22), datetime.date(2026, 8, 23)), (10, datetime.date(2026, 11, 17), datetime.date(2026, 11, 20), datetime.date(2026, 11, 20))]
Item atualizado: [(Decimal('60.00'), Decimal('10.10'), 10, 10, 10)]


## 5. Menu Interativo do Sistema
Execute a célula abaixo para operar o sistema via terminal interativo (`input`).

In [101]:
def menu_confeitaria():
    while True:
        print("\n=== SISTEMA CONFEITARIA ===")
        print("1. Cadastrar Cliente")
        print("2. Listar Clientes")
        print("3. Atualizar Endereço de Cliente")
        print("4. Remover Cliente")
        print("5. Cadastrar Tipo de Doce")
        print("6. Listar Tipos de Doce")
        print("7. Cadastrar Doce")
        print("8. Listar Doces")
        print("9. Registrar Pedido e Item")
        print("10. Listar Pedidos e Itens")
        print("0. Sair")

        opcao = input("Escolha uma opção: ").strip()

        if opcao == "1":
            nome = input("Nome: ")
            logradouro = input("Logradouro: ")
            numero = int(input("Número: "))
            bairro = input("Bairro: ")
            cidade = input("Cidade: ")
            cep = input("CEP (ex: 90000-000): ")
            criado = criar_cliente(nome, logradouro, numero, bairro, cidade, cep)
            if criado:
                print("Cadastrado:", criado)

        elif opcao == "2":
            for c in ler_clientes():
                print(c)

        elif opcao == "3":
            id_c = int(input("ID do cliente: "))
            log = input("Novo logradouro: ")
            num = int(input("Novo número: "))
            bai = input("Novo bairro: ")
            cid = input("Nova cidade: ")
            cep = input("Novo CEP: ")
            if atualizar_cliente(id_c, log, num, bai, cid, cep):
                print("Cliente atualizado com sucesso!")
            else:
                print("Cliente não encontrado.")

        elif opcao == "4":
            id_c = int(input("ID do cliente para remover: "))
            if deletar_cliente(id_c):
                print("Cliente removido!")
            else:
                print("Não foi possível remover (verifique se existe ou se possui vínculos).")

        elif opcao == "5":
            sabor = input("Sabor: ")
            tipo = input("Tipo (ex: Bolo, Torta, Cupcake): ")
            criado = criar_tipo_doce(sabor, tipo)
            if criado:
                print("Tipo cadastrado:", criado)

        elif opcao == "6":
            for t in ler_tipos_doce():
                print(t)

        elif opcao == "7":
            id_t = int(input("ID do Tipo de Doce: "))
            preco = float(input("Preço (R$): "))
            criado = criar_doce(id_t, preco)
            if criado:
                print("Doce cadastrado:", criado)

        elif opcao == "8":
            for d in ler_doces():
                print(d)

        elif opcao == "9":
            id_c = int(input("ID do Cliente: "))
            id_d = int(input("ID do Doce: "))
            dt_feita = input("Data do pedido (AAAA-MM-DD): ")
            dt_prev = input("Data prevista (AAAA-MM-DD): ")
            pago = float(input("Total pago (R$): "))
            troco = float(input("Troco (R$): "))
            ped = criar_pedido(dt_feita, dt_prev, None)
            if ped:
                criar_solicitacao(id_c, ped.id_pedido)
                item = criar_item_pedido(ped.id_pedido, id_d, id_c, pago, troco)
                print("Pedido registrado:", ped, "|", item)

        elif opcao == "10":
            print("--- Pedidos ---")
            for p in ler_pedidos():
                print(p)
            print("--- Itens ---")
            for i in ler_itens_pedido():
                print(i)

        elif opcao == "0":
            print("Encerrando menu...")
            break
        else:
            print("Opção inválida!")

# Descomente a linha abaixo para iniciar o menu interativo:
# menu_confeitaria()